# RDataFrame features for analysis workflows

This notebook collects a few practical RDataFrame features that are useful after you already know the basic workflow:

```text
RDataFrame -> Define -> Filter -> Histo1D / Count / Mean
```

The goal here is not to introduce every advanced feature, but to show tools that appear often in real analyses:

- saving a processed dataset with `Snapshot`
- producing a cut-flow report with named filters
- using small C++ helper functions from Python
- handling multiple input files or samples
- redefining an existing column


## Saving a processed dataset with `Snapshot`

So far, we have mostly used actions such as `Histo1D`, `Count`, or `Mean`.

Another common action is `Snapshot`. It writes selected columns of an RDataFrame to a new ROOT file.

Typical use cases:

- save a reduced ntuple after preselection,
- save newly computed variables,
- keep only the columns needed for a later statistical analysis.

The important point is:

> `Snapshot` is an **action**. By default, it triggers the event loop.

In [ ]:
import ROOT

df = ROOT.RDataFrame("Events", "data/example_file.root")
df_with_c = df.Define("c", "a + b")

# Output configuration
out_treename = "Events"
out_filename = "outtree.root"
out_columns = ["a", "b", "c"]

# Save selected columns to a new ROOT file.
snapdf = df_with_c.Snapshot(out_treename, out_filename, out_columns)

We can inspect the output file with the ROOT command-line tool `rootls`.

The option `-l` prints details, and `-t` also prints the tree structure.

In [ ]:
%%bash
rootls -lt outtree.root

The result returned by `Snapshot` is itself an RDataFrame connected to the output tree. This allows you to immediately inspect or continue processing the saved dataset.

In [ ]:
snapdf.Display().Print()

### Lazy `Snapshot`

Sometimes you want to define a `Snapshot`, but not run it immediately. This is useful when you want several actions to be executed in the same event loop.

This is controlled via `ROOT.RDF.RSnapshotOptions`.

In [ ]:
options = ROOT.RDF.RSnapshotOptions()

# Do not trigger the event loop immediately.
# The Snapshot will run only when a result is requested.
options.fLazy = True

# Preserve std::vector columns as std::vector in the output file.
# This option matters for some downstream workflows.
options.fVector2RVec = False

lazy_snapshot = df_with_c.Snapshot(out_treename, "outtree_lazy.root", out_columns, options)
# At this point, because fLazy=True, the event loop has not necessarily run yet.

## Cut-flow reports

In many analyses, you apply a sequence of selections:

```text
all events
  -> require one lepton
  -> require missing energy
  -> require two jets
  -> signal region
```

A cut-flow report tells you how many events pass each step.

In RDataFrame, this is done by giving names to filters and then calling `Report()`.

In [ ]:
df = ROOT.RDataFrame("sig_tree", "https://root.cern/files/Higgs_data.root")

selected = (
    df
    .Filter("lepton_eta > 0", "positive lepton eta")
    .Filter("lepton_phi < 1", "lepton phi < 1")
)

report = selected.Report()
report.Print()

Notice that the report is requested from the **last node** of the computation graph (`selected`), not from the original dataframe.

This way the report includes the named filters that are part of the analysis chain.

A useful pattern is:

```python
selected = (
    df
    .Filter("cut1", "name of cut 1")
    .Filter("cut2", "name of cut 2")
)

hist = selected.Histo1D("variable")
report = selected.Report()
```

Then the histogram and report describe the same selection.

## Using C++ helper functions from Python

RDataFrame can be used from Python, but the expressions passed to `Define` and `Filter` are usually C++ expressions.

For simple operations, this is enough:

```python
df.Define("z", "x + y")
```

For more complex operations, it is often better to define a small C++ helper function and call it from RDataFrame.
This mechanism uses the C++ interpreter `cling` shipped with ROOT, making this possible in a single line of code.
This has two advantages:

- the code runs as compiled C++;
- the function can be used safely in multi-threaded RDataFrame, provided it has no shared mutable state.

In a notebook, we can declare C++ code with the `%%cpp` magic.


Let's start by defining a function that will allow us to change the type of a the RDataFrame dataset entry numbers (stored in the special column "rdfentry") from `unsigned long long` to `float`.


In [ ]:
%%cpp

float asfloat(unsigned long long entrynumber) {
    return entrynumber;
}

In [ ]:
%%cpp
float square(float x) {
    return x * x;
}

Now we create a small artificial dataset with 100 entries.

The special column `rdfentry_` contains the entry number. It exists automatically in RDataFrame.

In [ ]:
df = ROOT.RDataFrame(100)

df1 = (
    df
    .Define("a", "asfloat(rdfentry_)")
    .Define("b", "square(a)")
)
df1.Display(["rdfentry_", "a", "b"], 5).Print()

The new columns can be used like any other columns. For example, we can make a graph of `b = a**2`.

In [ ]:
canvas = ROOT.TCanvas()
graph = df1.Graph("a", "b")

graph.SetMarkerStyle(20)
graph.SetMarkerSize(0.5)
graph.SetTitle("Example graph; a; b = a^{2}")

graph.Draw("AP")
canvas.Draw()

## Redefining a column

Normally, `Define` creates a new column name:

```python
df.Define("y", "x * 10")
```

Sometimes you want to keep the same column name but change its values. For this, use `Redefine`.

This is useful for simple corrections or transformations where downstream code expects the original column name.

In [ ]:
df = ROOT.RDataFrame(15).Define("x", "42")

df.Display(["x"], nRows=5).Print()

In [ ]:
df_redefined = df.Redefine("x", "x * 10")

df_redefined.Display(["x"], nRows=5).Print()

`Redefine` can use the previous value of the column to compute the new value.

Use it carefully: after redefining a column, the original value is no longer available under the same name. If you want to keep both, prefer:

```python
df.Define("x_corrected", "x * 10")
```

## Multiple input files and per-sample information

Real analyses usually read many files.

The simplest case is to pass a list of files to one RDataFrame:

```python
df = ROOT.RDataFrame("Events", ["file1.root", "file2.root"])
```

RDataFrame treats the input as one logical dataset.

Sometimes you also want to know which sample a given event came from, for example to assign a sample-dependent weight. For that use case, RDataFrame provides `DefinePerSample`.

In [ ]:
def create_sample(idx, sample_name):
    (
        ROOT.RDataFrame(5)
        .Define("x", f"42 + {idx}")
        .Snapshot("Events", sample_name)
    )

samples = ["sample_A.root", "sample_B.root"]

for idx, sample in enumerate(samples):
    create_sample(idx, sample)

df = ROOT.RDataFrame("Events", samples)

df_with_weight = df.DefinePerSample(
    "weight",
    'rdfsampleinfo_.Contains("sample_A") ? 11 : 22'
)

df_with_weight.Display(["x", "weight"], 10).Print()

The expression above assigns:

- weight `11` for events coming from `sample_A.root`,
- weight `22` for events coming from `sample_B.root`.

In a real analysis, this could represent:

- luminosity weights,
- cross-section weights,
- data/MC labels,
- year-dependent constants.

## RDatsetSpec and FromSpec

What if you have many different samples - multiple tree names and many different file names with some metadata information as well? 
- We have the RDatasetSpec class for that!
- And especially, get familiar with the `FromSpec` method thanks to which you can create a single dataframe consisting of many different samples which are added via a JSON file.

Example:
```json
{
   "samples": {
      "sampleA": {
         "trees": ["tree1", "tree2"],
         "files": ["file1.root", "file2.root"],
         "metadata": {"lumi": 1.0, }
      },
      "sampleB": {
         "trees": ["tree3", "tree4"],
         "files": ["file3.root", "file4.root"],
         "metadata": {"lumi": 0.5, }
      },
      ...
    },
}
```

In [ ]:
df = ROOT.RDF.Experimental.FromSpec("data/dataset_spec.json")

But what if some operations are valid only for some of the samples? 
- Solution: use `DefinePerSample` method - the samples can be recognized thanks to one or more metadata instances and then we can make use of `RSampleInfo` class. 

In [ ]:
df = df.DefinePerSample("xsecs", 'rdfsampleinfo_.GetD("xsecs")')